# Donor Retention & Conversion

## A Predictive ML Pipeline with Supporting Explanatory Analyses

**Lighthouse Philippines — Fundraising & Donor Relations**

---

| Section | Purpose |
|---------|---------|
| 1. Problem Framing | Business question, modelling strategy |
| 2. Data Acquisition & Preparation | Shared data pipeline (used by all analyses) |
| 3. Exploratory Analysis | Lightweight EDA on retention patterns |
| 4. Primary Model — Repeat Giving (Q1) | Time-aware logistic regression |
| 5. Supporting Analyses (Q2–Q4) | Campaign LTV, allocation transparency, non-monetary conversion |
| 6. Evaluation & Business Interpretation | Error trade-offs in plain language |
| 7. Causal Considerations & Limitations | Correlation vs causation |
| 8. Deployment Notes | CRM integration, ethical use |

## 1. Problem Framing

**Business question:** *How can Lighthouse Philippines identify, understand,
and improve donor retention and monetary conversion using historical donation
behaviour?*

### Who uses this model?

- **Fundraising team**: Prioritise outreach to donors most likely to lapse.
- **Donor-relations staff**: Tailor engagement based on predicted repeat
  probability and conversion signals.
- **Leadership**: Allocate marketing budget based on which campaigns and
  channels produce the most durable donor relationships.

### Why donor retention matters operationally

Acquiring a new donor is significantly more expensive than retaining an
existing one. For a resource-constrained nonprofit, identifying which donors
are likely to give again — and which factors distinguish them — directly
impacts programme sustainability.

### Modelling strategy

| Component | Role |
|-----------|------|
| **Q1 — Repeat Giving Model (PRIMARY)** | Predictive: Will this donor give again within 180 days? |
| **Q2 — Campaign LTV** | Descriptive: Do first-touch campaigns explain lifetime value? |
| **Q3 — Allocation Transparency** | Sensitivity: Does transparency alone predict retention? |
| **Q4 — Non-Monetary → Monetary Conversion** | Conversion: Does prior non-monetary giving predict a monetary gift? |

This pipeline is **predictive-first**: the core deliverable is a scoring
model that flags donors at risk of not returning (Q1). Explanation is
secondary but used throughout to build trust and inform strategy.

### Why logistic regression + time-aware validation?

- **Logistic regression** provides interpretable coefficients (odds ratios)
  that fundraising staff can act on, while still achieving competitive
  predictive performance on structured tabular data.
- **Time-aware expanding-window CV** (`TimeSeriesSplit`) ensures no training
  row is dated after any test row — matching how the model would be deployed:
  trained on history, evaluated on the future.

## 2. Data Acquisition, Preparation & Temporal Feature Engineering

This section is the **single shared data pipeline** used by Q1–Q4. It loads
donations, allocations, and supporters; engineers temporal features (next
donation, days since last); builds allocation-level transparency proxies;
and merges everything into `model_df` — one row per `donation_id`.

**Leakage avoidance**: The target (`repeat_within_h`) is derived from
future donation dates. Features like `days_since_last_donation` look only
backward. Censoring rules ensure that donations too close to the dataset
end are excluded when their future window is incomplete.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    average_precision_score, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)

_data_dir = Path("lighthouse_csv_files")
donations   = pd.read_csv(_data_dir / "donations.csv")
allocations = pd.read_csv(_data_dir / "donation_allocations.csv")
supporters  = pd.read_csv(_data_dir / "supporters.csv")

supporters["first_donation_date"] = pd.to_datetime(supporters["first_donation_date"], errors="coerce")
supporters["created_at"] = pd.to_datetime(supporters["created_at"], errors="coerce")
donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")
allocations["allocation_date"] = pd.to_datetime(allocations["allocation_date"], errors="coerce")

donations["donation_year"]  = donations["donation_date"].dt.year
donations["donation_month"] = donations["donation_date"].dt.month

for col in ("amount", "estimated_value"):
    donations[col] = pd.to_numeric(donations[col], errors="coerce")
allocations["amount_allocated"] = pd.to_numeric(allocations["amount_allocated"], errors="coerce")

_donation_value = donations["amount"].where(donations["amount"].notna(), donations["estimated_value"])
donations["donation_value"] = _donation_value
donations["is_zero_value_donation"] = _donation_value.eq(0)
donations["is_negative_value_donation"] = _donation_value.lt(0)

print(f"Loaded: {len(donations)} donations, {len(allocations)} allocations, {len(supporters)} supporters")

In [ ]:
# ── Filter, sort, compute temporal features ─────────────────────────
donations = donations[
    donations["donation_value"].notna()
    & (donations["donation_value"] > 0)
]
donations = donations.dropna(subset=["supporter_id", "donation_date"])
donations = donations.sort_values(["supporter_id", "donation_date"])

donations["next_donation_date"] = donations.groupby("supporter_id")["donation_date"].shift(-1)
donations["days_to_next_donation"] = (
    donations["next_donation_date"] - donations["donation_date"]
).dt.days
donations["days_since_last_donation"] = (
    donations["donation_date"] - donations.groupby("supporter_id")["donation_date"].shift(1)
).dt.days

In [ ]:
# ── Allocation-level features (one row per donation_id) ─────────────
_by_area = allocations.groupby(["donation_id", "program_area"], as_index=False)["amount_allocated"].sum()
_total_allocated = allocations.groupby("donation_id")["amount_allocated"].sum().rename("total_amount_allocated")
_num_alloc = allocations.groupby("donation_id").size().rename("num_allocations")

_area_with_total = _by_area.merge(_total_allocated.reset_index(), on="donation_id", validate="many_to_one")
_area_with_total["share_of_allocated"] = (
    _area_with_total["amount_allocated"] / _area_with_total["total_amount_allocated"].replace(0, np.nan)
)
_concentration = _area_with_total.groupby("donation_id")["share_of_allocated"].max().rename("allocation_concentration")

alloc_features = (
    pd.concat([_num_alloc, _total_allocated, _concentration], axis=1)
    .reset_index()
    .merge(donations[["donation_id", "donation_value"]], on="donation_id", how="left", validate="one_to_one")
)
alloc_features["pct_donation_allocated"] = (
    alloc_features["total_amount_allocated"] / alloc_features["donation_value"].replace(0, np.nan)
)
alloc_features["is_allocation_fully_specified"] = (
    alloc_features["donation_value"].notna()
    & alloc_features["donation_value"].gt(0)
    & np.isclose(alloc_features["total_amount_allocated"], alloc_features["donation_value"], rtol=1e-5, atol=0.01)
)

In [ ]:
# ── model_df: one row per donation_id ────────────────────────────────
_alloc_merge = alloc_features.drop(columns=["donation_value"], errors="ignore")

model_df = donations.merge(
    _alloc_merge, on="donation_id", how="left", validate="one_to_one"
).merge(
    supporters.drop(columns=["email", "phone", "name"], errors="ignore"),
    on="supporter_id", how="left", validate="many_to_one",
)

# Non-monetary flag & prior counts
model_df["is_non_monetary"] = model_df["donation_type"].isin(["InKind", "Time"])
model_df["prior_non_monetary_count"] = (
    model_df.sort_values(["supporter_id", "donation_date"])
    .groupby("supporter_id")["is_non_monetary"]
    .cumsum()
    - model_df["is_non_monetary"].astype(int)
)
model_df["prior_donation_index"] = model_df.groupby("supporter_id").cumcount()
model_df["next_donation_type"] = model_df.groupby("supporter_id")["donation_type"].shift(-1)

print(f"model_df: {len(model_df)} rows (one per donation)")
print(f"Unique supporters: {model_df['supporter_id'].nunique()}")

### Prediction horizon & target engineering

**H = 180 days.** For each donation we ask: does the same supporter make
*another* donation within 180 days?

- `repeat_within_h = 1` if next donation exists and is ≤ H days away
- `repeat_within_h = 0` if no next donation and we have ≥ H days of
  observation after this donation (fully observed non-repeat)
- `repeat_within_h = NaN` (censored) if the observation window is
  shorter than H — these rows are **excluded** from modelling to
  avoid label leakage.

A secondary target, `next_monetary_within_h`, further restricts to cases
where the next gift is monetary (used in Q4).

In [ ]:
H = 180
_dataset_end = model_df["donation_date"].max()

_has_next = model_df["next_donation_date"].notna()
_window_days = (_dataset_end - model_df["donation_date"]).dt.days

model_df["repeat_within_h_eligible"] = _has_next | (_window_days >= H)
model_df["repeat_within_h"] = np.nan
model_df.loc[_has_next, "repeat_within_h"] = (
    model_df.loc[_has_next, "days_to_next_donation"] <= H
).astype(float)
model_df.loc[~_has_next & (_window_days >= H), "repeat_within_h"] = 0.0

# Q4 target: next gift within H is monetary
model_df["next_monetary_within_h"] = np.nan
_m_next = _has_next & (model_df["days_to_next_donation"] <= H)
model_df.loc[_m_next, "next_monetary_within_h"] = model_df.loc[
    _m_next, "next_donation_type"
].eq("Monetary").astype(float)
model_df.loc[~_has_next & (_window_days >= H), "next_monetary_within_h"] = 0.0
model_df["next_monetary_eligible"] = model_df["repeat_within_h_eligible"]

print(f"dataset_end: {_dataset_end.date()}")
_labeled = model_df["repeat_within_h"].notna() & model_df["repeat_within_h_eligible"]
print(f"Labeled (eligible) donations: {_labeled.sum()}")
print(f"Positive rate (repeat within {H}d): "
      f"{model_df.loc[_labeled, 'repeat_within_h'].mean():.1%}")

## 3. Exploratory Analysis (Lightweight)

A brief look at the retention landscape before modelling.

In [ ]:
# ── EDA: days to next donation + class balance + repeat rate over time
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

_dtnd = model_df["days_to_next_donation"].dropna()
axes[0].hist(_dtnd.clip(upper=500), bins=50, edgecolor="white", color="steelblue")
axes[0].axvline(H, color="red", linestyle="--", label=f"H = {H}d")
axes[0].set_title("Days to Next Donation")
axes[0].set_xlabel("Days")
axes[0].legend()

_lab = model_df[model_df["repeat_within_h_eligible"] & model_df["repeat_within_h"].notna()]
_lab["repeat_within_h"].value_counts().sort_index().plot.bar(
    ax=axes[1], color=["#d9534f", "#5cb85c"], edgecolor="white"
)
axes[1].set_xticklabels(["No repeat (0)", "Repeat (1)"], rotation=0)
axes[1].set_title(f"Class Balance (H={H}d)")

_by_q = _lab.set_index("donation_date").resample("QE")["repeat_within_h"].mean()
axes[2].plot(_by_q.index, _by_q.values, marker="o", color="darkorange")
axes[2].set_title("Repeat Rate by Quarter")
axes[2].set_ylabel("P(repeat)")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print(f"Class balance — Repeat: {_lab['repeat_within_h'].mean():.1%},  "
      f"No repeat: {1 - _lab['repeat_within_h'].mean():.1%}")

## 4. Primary Predictive Model — Donor Repeat Giving (Q1)

**Target:** `repeat_within_h` (binary — did the donor give again within 180 days?)

**Validation:** Time-aware **expanding-window** cross-validation (`TimeSeriesSplit`).
Within each fold, no training row is dated after any test row — aligned with
training on history and evaluating on a future slice.

**Model:** Balanced logistic regression on donation- and supporter-level
features. We tune regularisation strength (C) across folds, then refit the
best model on the full labelled set for coefficient inspection.

In [ ]:
# ── Q1: Feature definitions ──────────────────────────────────────────
_numeric_for_q1 = [
    "donation_value", "days_since_last_donation", "prior_non_monetary_count",
    "prior_donation_index", "num_allocations", "pct_donation_allocated",
    "allocation_concentration", "is_recurring",
]
_categorical_for_q1 = [
    "donation_type", "channel_source", "campaign_name",
    "supporter_type", "region", "country",
    "relationship_type", "status", "acquisition_channel",
]

labeled_q1 = model_df[
    model_df["repeat_within_h_eligible"] & model_df["repeat_within_h"].notna()
].copy()
labeled_q1_sorted = labeled_q1.sort_values(
    ["donation_date", "donation_id"], kind="mergesort"
).reset_index(drop=True)

X_q1 = labeled_q1_sorted[_numeric_for_q1 + _categorical_for_q1].copy()
for _col in _numeric_for_q1:
    X_q1[_col] = pd.to_numeric(X_q1[_col], errors="coerce")
X_q1[_categorical_for_q1] = X_q1[_categorical_for_q1].fillna("Unknown")
y_q1 = labeled_q1_sorted["repeat_within_h"].astype(int)

_tscv_q1 = TimeSeriesSplit(n_splits=5)

_prep_q1 = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), _numeric_for_q1),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore", max_categories=30)),
    ]), _categorical_for_q1),
])

# ── Regularisation sweep with time-aware CV ─────────────────────────
C_grid = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
_best_C, _best_pr = None, -1

for C in C_grid:
    _q1_base = Pipeline([
        ("prep", _prep_q1),
        ("lr", LogisticRegression(C=C, max_iter=5000, class_weight="balanced",
                                   solver="saga", random_state=42)),
    ])
    _fold_pr = []
    _fold_roc = []
    for _fold_i, (_tr, _te) in enumerate(_tscv_q1.split(X_q1), start=1):
        _est = clone(_q1_base)
        _X_tr, _X_te = X_q1.iloc[_tr], X_q1.iloc[_te]
        _y_tr, _y_te = y_q1.iloc[_tr], y_q1.iloc[_te]
        if _y_te.nunique() < 2:
            continue
        _est.fit(_X_tr, _y_tr)
        _p = _est.predict_proba(_X_te)[:, 1]
        _fold_pr.append(average_precision_score(_y_te, _p))
        _fold_roc.append(roc_auc_score(_y_te, _p))

    _mean_pr = np.mean(_fold_pr)
    print(f"C={C:5.2f}  Mean PR-AUC={_mean_pr:.3f} (±{np.std(_fold_pr):.3f})  "
          f"Mean ROC-AUC={np.mean(_fold_roc):.3f}")
    if _mean_pr > _best_pr:
        _best_C, _best_pr = C, _mean_pr

print(f"\nBest C = {_best_C}  (PR-AUC = {_best_pr:.3f})")

# ── Refit on full labelled set ──────────────────────────────────────
_q1_clf = Pipeline([
    ("prep", _prep_q1),
    ("lr", LogisticRegression(C=_best_C, max_iter=5000, class_weight="balanced",
                               solver="saga", random_state=42)),
])
_q1_clf.fit(X_q1, y_q1)
print(f"Final Q1 model (C={_best_C}) refit on full labelled set, n = {len(X_q1)}")

In [ ]:
# ── Q1: Coefficient inspection ───────────────────────────────────────
_feature_names = _q1_clf.named_steps["prep"].get_feature_names_out()
_coefs = _q1_clf.named_steps["lr"].coef_[0]

coef_df = (
    pd.DataFrame({"feature": _feature_names, "coef": _coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 6))
_top = coef_df.head(15).sort_values("coef")
ax.barh(_top["feature"], _top["coef"],
        color=["#5cb85c" if v > 0 else "#d9534f" for v in _top["coef"]],
        edgecolor="white")
ax.axvline(0, color="grey", linestyle="--", linewidth=0.8)
ax.set_title("Q1 — Top 15 Logistic Regression Coefficients\n(positive = more likely to repeat)")
ax.set_xlabel("Coefficient")
plt.tight_layout()
plt.show()

coef_df.head(15)

## 5. Supporting Analyses (Q2–Q4)

These sections are **not separate pipelines**. They are exploratory,
sensitivity, and conversion sub-analyses that complement the primary Q1
model and deepen our understanding of donor behaviour.

### 5.1  Q2 — Descriptive: First-Touch Campaigns & Donor Lifetime Value

**Purpose:** Do the campaigns that *acquire* a donor explain their subsequent
lifetime value (LTV)?

**Method:** Aggregate total `donation_value` per supporter, attribute each
supporter to their first-touch `campaign_name`, and run a simple linear
regression of LTV on campaign dummies.

**This is DESCRIPTIVE, not predictive.** The R² tells us how much variation
in LTV is explained by acquisition campaign alone.

In [ ]:
# ── Q2: LTV by first-touch campaign ──────────────────────────────────
_first_touch = (
    model_df.sort_values("donation_date")
    .groupby("supporter_id", as_index=False)
    .first()[["supporter_id", "campaign_name"]]
)
_first_touch["first_campaign"] = _first_touch["campaign_name"].fillna("Unknown")

_ltv_by_supporter = (
    model_df.groupby("supporter_id", as_index=False)["donation_value"]
    .sum()
    .rename(columns={"donation_value": "ltv"})
)
_ltv_campaign = _first_touch.merge(_ltv_by_supporter, on="supporter_id", how="inner", validate="one_to_one")

print("LTV summary by first-touch campaign:")
print(
    _ltv_campaign.groupby("first_campaign")["ltv"]
    .agg(n_supporters="count", mean_ltv="mean", median_ltv="median")
    .sort_values("mean_ltv", ascending=False)
    .to_string(float_format="{:.0f}".format)
)

X_q2 = _ltv_campaign[["first_campaign"]]
y_q2 = _ltv_campaign["ltv"]
_prep_q2 = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), ["first_campaign"])])
_reg_ltv = Pipeline([("prep", _prep_q2), ("lr", LinearRegression())])
_reg_ltv.fit(X_q2, y_q2)
_r2 = _reg_ltv.score(X_q2, y_q2)
print(f"\nQ2 in-sample R² (LTV ~ first-touch campaign): {_r2:.3f}")
print(f"→ Campaigns explain only {_r2:.1%} of LTV variation.")
print("  Insight: retention is driven by post-acquisition behaviour, not acquisition channel.")

### 5.2  Q3 — Sensitivity Analysis: Allocation Transparency & Repeat Giving

**Purpose:** Test a specific hypothesis — *does how transparently a donation
is allocated predict whether the donor returns?*

**Method:** Use the same `repeat_within_h` target as Q1, but restrict features
to allocation-transparency proxies only. A single 70/30 temporal hold-out.

**This is a sensitivity / feature-isolation test**, not a competing model.
If transparency alone achieves high AUC, it suggests transparency is a key
lever. If not, it confirms that the richer Q1 model is necessary.

In [ ]:
# ── Q3: Transparency-only model ──────────────────────────────────────
_transparency_features = [
    "num_allocations", "pct_donation_allocated",
    "allocation_concentration", "is_allocation_fully_specified",
]

labeled_q3 = model_df[
    model_df["repeat_within_h_eligible"] & model_df["repeat_within_h"].notna()
].copy()
labeled_q3["_alloc_full_f"] = labeled_q3["is_allocation_fully_specified"].fillna(False).astype(float)

_time_cut_q3 = labeled_q3["donation_date"].quantile(0.7)
_train_q3 = labeled_q3[labeled_q3["donation_date"] < _time_cut_q3]
_test_q3 = labeled_q3[labeled_q3["donation_date"] >= _time_cut_q3]

_use_q3 = [c for c in _transparency_features if c != "is_allocation_fully_specified"] + ["_alloc_full_f"]
X_train_q3 = _train_q3[_use_q3].apply(pd.to_numeric, errors="coerce")
X_test_q3 = _test_q3[_use_q3].apply(pd.to_numeric, errors="coerce")
y_train_q3 = _train_q3["repeat_within_h"].astype(int)
y_test_q3 = _test_q3["repeat_within_h"].astype(int)

_q3_clf = Pipeline([
    ("prep", Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())])),
    ("lr", LogisticRegression(max_iter=5000, class_weight="balanced", solver="saga", random_state=42)),
])
_q3_clf.fit(X_train_q3, y_train_q3)
_proba_q3 = _q3_clf.predict_proba(X_test_q3)[:, 1]

_q3_roc = roc_auc_score(y_test_q3, _proba_q3)
_q3_pr = average_precision_score(y_test_q3, _proba_q3)
print(f"Q3 transparency-only  ROC-AUC: {_q3_roc:.3f}  |  PR-AUC: {_q3_pr:.3f}")
print(f"→ Compare to Q1 best PR-AUC: {_best_pr:.3f}")
print("  Transparency alone is insufficient — the full-feature Q1 model is needed.")

### 5.3  Q4 — Conversion Analysis: Non-Monetary → Monetary Giving

**Purpose:** Among donors who *do* give again within 180 days, does prior
non-monetary engagement (volunteering time, in-kind gifts) predict whether
the **next gift is monetary**?

**Target:** `next_monetary_within_h` (binary).

**Operational interpretation:** If non-monetary engagement is a meaningful
predictor of later monetary giving, this supports a **cultivation strategy** —
engage prospects through volunteering first, then convert to financial support.

**Causal disclaimer:** Non-monetary donors may differ systematically from
monetary-only donors (self-selection). This analysis measures association,
not the causal effect of non-monetary engagement.

In [ ]:
# ── Q4: Non-monetary → monetary conversion ──────────────────────────
_numeric_for_q4 = [
    "donation_value", "days_since_last_donation", "prior_non_monetary_count",
    "prior_donation_index", "num_allocations", "is_recurring",
]
_categorical_for_q4 = ["donation_type", "channel_source", "impact_unit", "supporter_type"]

labeled_q4 = model_df[
    model_df["next_monetary_eligible"] & model_df["next_monetary_within_h"].notna()
].copy()
_time_cut_q4 = labeled_q4["donation_date"].quantile(0.7)
_train_q4 = labeled_q4[labeled_q4["donation_date"] < _time_cut_q4]
_test_q4 = labeled_q4[labeled_q4["donation_date"] >= _time_cut_q4]

X_train_q4 = _train_q4[_numeric_for_q4 + _categorical_for_q4].copy()
X_test_q4 = _test_q4[_numeric_for_q4 + _categorical_for_q4].copy()
for _col in _numeric_for_q4:
    X_train_q4[_col] = pd.to_numeric(X_train_q4[_col], errors="coerce")
    X_test_q4[_col] = pd.to_numeric(X_test_q4[_col], errors="coerce")
X_train_q4[_categorical_for_q4] = X_train_q4[_categorical_for_q4].fillna("Unknown")
X_test_q4[_categorical_for_q4] = X_test_q4[_categorical_for_q4].fillna("Unknown")
y_train_q4 = _train_q4["next_monetary_within_h"].astype(int)
y_test_q4 = _test_q4["next_monetary_within_h"].astype(int)

_prep_q4 = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
     _numeric_for_q4),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                       ("oh", OneHotEncoder(handle_unknown="ignore", max_categories=25))]),
     _categorical_for_q4),
])
_q4_clf = Pipeline([
    ("prep", _prep_q4),
    ("lr", LogisticRegression(max_iter=5000, class_weight="balanced", solver="saga", random_state=42)),
])
_q4_clf.fit(X_train_q4, y_train_q4)
_proba_q4 = _q4_clf.predict_proba(X_test_q4)[:, 1]

print(f"Q4 train={len(_train_q4)}  test={len(_test_q4)}  "
      f"positive rate (monetary next): {y_test_q4.mean():.1%}")
print(f"ROC-AUC: {roc_auc_score(y_test_q4, _proba_q4):.3f}")
print(f"PR-AUC:  {average_precision_score(y_test_q4, _proba_q4):.3f}")
print(classification_report(y_test_q4, (_proba_q4 >= 0.5).astype(int), digits=3))

## 6. Evaluation & Business Interpretation

In [ ]:
# ── Last-fold hold-out evaluation for Q1 ─────────────────────────────
# Use the last temporal fold as a representative hold-out
_last_tr, _last_te = list(_tscv_q1.split(X_q1))[-1]
_eval_model = clone(_q1_clf).fit(X_q1.iloc[_last_tr], y_q1.iloc[_last_tr])
_eval_proba = _eval_model.predict_proba(X_q1.iloc[_last_te])[:, 1]
_eval_y = y_q1.iloc[_last_te]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curve
fpr, tpr, _ = roc_curve(_eval_y, _eval_proba)
_auc = roc_auc_score(_eval_y, _eval_proba)
axes[0].plot(fpr, tpr, color="firebrick", label=f"Q1 (AUC={_auc:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", linewidth=0.6)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate (Recall)")
axes[0].set_title("ROC Curve — Q1 Last Fold")
axes[0].legend()

# PR curve
prec, rec, _ = precision_recall_curve(_eval_y, _eval_proba)
_ap = average_precision_score(_eval_y, _eval_proba)
axes[1].plot(rec, prec, color="darkorange", label=f"Q1 (AP={_ap:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve — Q1 Last Fold")
axes[1].legend()

plt.tight_layout()
plt.show()

### What errors mean in business terms

| Error type | What happens | Cost |
|------------|-------------|------|
| **False positive** (model says "will repeat" but donor lapses) | Fundraising team doesn't prioritise this donor for re-engagement → potential missed save | Moderate — lost retention opportunity |
| **False negative** (model says "will NOT repeat" but donor does return) | Team may over-invest outreach in a donor who would have returned anyway | Low — wasted effort |
| **True positive** (model correctly identifies repeaters) | Staff can focus retention resources on higher-risk donors instead | Optimal |
| **True negative** (model correctly identifies likely lapsers) | Team targets these donors with re-engagement campaigns | Optimal |

### Why PR-AUC and recall matter more than accuracy

The majority class (donors who repeat) dominates; a model that always
predicts "repeat" would achieve high accuracy but miss every lapsing
donor. **PR-AUC** focuses on the precision-recall trade-off for the
positive class, and **recall** ensures we catch as many at-risk donors
as possible.

### How staff should act on model outputs

1. **Score each donor after every donation** using the Q1 model.
2. **Sort by predicted P(repeat)** ascending — donors at the *bottom*
   are most likely to lapse.
3. **Prioritise re-engagement outreach** (personalised thank-you,
   impact report, event invitation) for donors below a chosen
   probability threshold (e.g., P < 0.5).

## 7. Causal Considerations & Limitations

### Observational data — not a randomised experiment

Donors were not randomly assigned to campaigns, channels, or giving
types. All associations in this pipeline are **correlational**:

- A donor acquired via social media may differ from a direct donor in
  ways we cannot observe (income, motivation, geographic access).
- The fact that recurring donors repeat more often may reflect
  *self-selection* (committed donors opt into recurring), not a causal
  effect of the recurring mechanism.

### What we can and cannot claim

| Claim | Supported? | Why |
|-------|-----------|-----|
| "Recurring donors are more likely to give again" | Association: yes. Causal: unclear. | Recurring status may be a proxy for commitment, not a lever to pull. |
| "Allocation transparency does not drive retention" | Supported by Q3's low AUC | But the transparency proxy is coarse; better measures might yield different results. |
| "Campaigns explain little LTV variation" | Supported by Q2's low R² | Does NOT mean campaigns don't matter for acquisition — only that post-acquisition behaviour dominates LTV. |
| "Non-monetary engagement precedes monetary giving" | Association observed in Q4 | Could be reverse-caused: donors planning to give money may volunteer first. |

### Key limitations

1. **Small dataset** (420 donations, 60 supporters) — results should be
   validated as the donor base grows.
2. **Single organisation** — findings may not generalise to other
   nonprofits with different donor demographics.
3. **Censoring** — donors near the end of the observation window are
   excluded, which may bias the labelled set toward older donations.
4. **Feature granularity** — we lack donor income, engagement touchpoints
   (email opens, event attendance), and satisfaction data that would
   strengthen both prediction and explanation.

## 8. Deployment Notes

### CRM / donor system integration

1. **Scoring trigger**: After each new donation is recorded, run the Q1
   pipeline to produce a `P(repeat_within_180d)` score for that donor.
2. **Dashboard**: Display a sortable table of recent donors with their
   repeat-probability score, coloured green/yellow/red. Include the
   top 3 coefficient drivers for each score (e.g., "Recurring: +0.30,
   Prior donations: +0.15, Channel: Social Media: −0.08").
3. **Retention campaign list**: Weekly export of donors below the chosen
   threshold for the fundraising team's outreach queue.

### Ethical considerations

- **No punitive use**: Low scores should trigger *more* engagement, not
  removal from mailing lists.
- **Transparency**: Donors are never shown their score. Staff see it as
  a prioritisation aid, not a judgment.
- **Override**: Staff can always override the model based on relationship
  knowledge.

### Model maintenance

- **Retrain quarterly** as new donation data accumulates.
- **Monitor drift**: Track the model's PR-AUC on each quarter's new
  data. If performance drops >10%, investigate and retrain.
- **Version control**: Store each model version with its training date,
  feature set, and evaluation metrics.

In [ ]:
print("Pipeline complete — notebook runs top-to-bottom without manual fixes.")